# PD Model Training – XGBoost (and tree-based baseline)

**Purpose:** Train a classical PD model on the engineered LendingClub features from **01_lendingclub_feature_engineering.ipynb**. We compare XGBoost with an optional tree-based baseline (e.g. Random Forest), then run hyperparameter tuning and export the best model for evaluation (`eval_runner` and proof JSONs).

**Input:** `data/credit_risk_pd/LendingClub/processed/lendingclub_engineered.parquet` (run notebook 01 first).

**Output:** Trained model saved to `models/pd/pd_model_local_v1.pkl` (or versioned path); metrics: AUC-ROC, F1, Precision, Recall.

**Tip:** If cells don't run, use **Kernel → Select Kernel** and pick the Python environment where you installed `pandas`, `xgboost`, `scikit-learn`. Run from the repo root folder (or open the repo root in VS Code/Cursor).

### Install packages into this kernel's environment
Run the cell below once to see **which Python** this notebook uses. Then in a terminal (PowerShell or Bash), run:
```
<that path> -m pip install -r requirements-notebooks.txt
```
from the repo root (or `notebooks/`). That way `pip` installs into the **same** environment as this kernel.

In [ ]:
import sys
print('This notebook kernel is using:')
print(sys.executable)
print()
print('To install packages into THIS environment, run in a terminal:')
print(f'  {sys.executable} -m pip install -r notebooks/requirements-notebooks.txt')
print('(from the repo root, or use a full path to requirements-notebooks.txt)')

## 1. Load engineered data and split

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Find repo root (works when run from repo root or from notebooks/)
ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "credit_risk").is_dir() and (ROOT / "data").is_dir():
        break
    ROOT = ROOT.parent
if not (ROOT / "credit_risk").is_dir():
    raise RuntimeError('Repo root not found. Run this notebook from the ocr-agentic-rag folder (or notebooks/ subfolder).')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from credit_risk.feature_engineering.common_features import get_feature_names

DATA_PATH = ROOT / "data" / "credit_risk_pd" / "LendingClub" / "processed" / "lendingclub_engineered.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_lendingclub_feature_engineering.ipynb first to create lendingclub_engineered.parquet")

df = pd.read_parquet(DATA_PATH)
feature_names = get_feature_names()
X = df[[c for c in feature_names if c in df.columns]]
y = df["default"]
if X.shape[1] != len(feature_names):
    for c in feature_names:
        if c not in X.columns:
            X[c] = 0.0
X = X[feature_names]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train {X_train.shape[0]:,} / Val {X_val.shape[0]:,}")

## 2. Iterative testing – XGBoost vs Random Forest

Train with default (or reasonable) hyperparameters to compare model families before tuning.

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

def eval_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc_roc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

results = {}

In [ ]:
import xgboost as xgb

scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
model_xgb = xgb.XGBClassifier(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale,
    random_state=42,
)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
p_val = model_xgb.predict_proba(X_val)[:, 1]
results["XGBoost"] = eval_binary(y_val, p_val)
print("XGBoost (default):", results["XGBoost"])

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced", random_state=42)
model_rf.fit(X_train, y_train)
p_val_rf = model_rf.predict_proba(X_val)[:, 1]
results["RandomForest"] = eval_binary(y_val, p_val_rf)
print("RandomForest (default):", results["RandomForest"])
print("\nComparison:", pd.DataFrame(results).T.round(4).to_string())

## 3. Hyperparameter tuning (XGBoost)

Use validation set (or cross-validation) to tune XGBoost; then refit on train+val if desired and save the final model.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "n_estimators": [80, 120, 150],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
}
search = RandomizedSearchCV(
    xgb.XGBClassifier(objective="binary:logistic", eval_metric="auc", scale_pos_weight=scale, random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV AUC:", search.best_score_.round(4))

In [ ]:
best_model = search.best_estimator_
p_val_best = best_model.predict_proba(X_val)[:, 1]
final_metrics = eval_binary(y_val, p_val_best)
print("Tuned XGBoost on validation:", final_metrics)

## 4. Save model for eval_runner

Persist the tuned model in the format expected by `credit_risk.models.pd_model.PDModel` (joblib with model, feature_names, params, metadata).

In [ ]:
import joblib

MODEL_DIR = ROOT / "models" / "pd"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_data = {
    "model": best_model,
    "feature_names": feature_names,
    "params": search.best_params_,
    "metadata": {
        "trained_at": pd.Timestamp.now().isoformat(),
        "n_features": len(feature_names),
        "n_train": len(X_train),
        "val_auc_roc": final_metrics["auc_roc"],
        "val_f1": final_metrics["f1"],
    },
}
path = MODEL_DIR / "pd_model_local_v1.pkl"
joblib.dump(model_data, path)
print(f"Saved to {path}")

## 5. Summary

- Trained XGBoost (and Random Forest) on engineered features; tuned XGBoost via `RandomizedSearchCV` (AUC-ROC).
- Best model saved as `models/pd/pd_model_local_v1.pkl`. Run `python eval_runner.py --category credit_risk_PD` to evaluate on LendingClub benchmark and generate proof JSONs.